# Stage 1: Non-Instruction Fine-Tuning
## Finance FAQ Assistant — Domain Adaptation on Raw Text

**Goal:** Adapt `Qwen2.5-0.5B` to finance vocabulary, terminology, and writing style *before* instruction tuning, by training it on raw domain paragraphs (next-token prediction, no instruction format).

**Runtime:** Google Colab, T4 GPU (Runtime → Change runtime type → T4 GPU)


## 1. Install dependencies

In [1]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()

!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes
!pip install datasets sentencepiece

## 2. Load raw domain text

In [2]:
# Upload data/non_instruction_data.txt to Colab, or mount Drive / clone the repo
# from google.colab import files
# uploaded = files.upload()  # select non_instruction_data.txt

DATA_PATH = "/content/data/non_instruction_data.txt"  # adjust path if using Drive/Git clone

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
print(f"Loaded {len(paragraphs)} paragraphs")
print(paragraphs[0][:300])

Loaded 57 paragraphs
A savings account is a deposit account held at a bank that earns interest on the balance. It is meant for individuals who want to set aside money safely while earning a modest return. Most savings accounts allow limited withdrawals per month and may charge a fee if the minimum balance is not maintai


## 3. Clean and chunk the text

In [3]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"[^\x00-\x7F]+", "", text)  # strip non-ASCII noise
    return text

cleaned_paragraphs = [clean_text(p) for p in paragraphs if len(p.split()) > 15]
print(f"{len(cleaned_paragraphs)} paragraphs after cleaning/length filter")

# Chunking strategy: each paragraph is already a coherent ~80-120 word chunk,
# which is a good size for causal-LM continuation training on a 0.5B model.
MAX_CHUNK_WORDS = 150

def chunk_paragraph(p, max_words=MAX_CHUNK_WORDS):
    words = p.split()
    if len(words) <= max_words:
        return [p]
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

chunks = []
for p in cleaned_paragraphs:
    chunks.extend(chunk_paragraph(p))

print(f"Final chunk count: {len(chunks)}")

57 paragraphs after cleaning/length filter
Final chunk count: 57


## 4. Load base model with Unsloth

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None  # auto-detect (bfloat16 on T4-supported hardware, float16 otherwise)
load_in_4bit = True  # QLoRA-style 4-bit loading to fit comfortably on T4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## 5. Apply LoRA (QLoRA via 4-bit base + LoRA adapters)

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                 # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.2 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## 6. Build dataset and train on raw text
Plain causal-LM continuation training — no instruction format, no chat template. The model simply learns to predict the next token in domain text, absorbing finance vocabulary and phrasing.

In [6]:
from datasets import Dataset

# Add EOS so the model learns natural paragraph boundaries
texts = [c + tokenizer.eos_token for c in chunks]
raw_dataset = Dataset.from_dict({"text": texts})
print(raw_dataset)

Dataset({
    features: ['text'],
    num_rows: 57
})


In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=raw_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs_stage1",
        save_strategy="no",
        report_to="none",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False}
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/57 [00:00<?, ? examples/s]

In [8]:
trainer_stats = trainer.train()
print(trainer_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 57 | Num Epochs = 3 | Total steps = 12
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,2.261713
10,1.997996


TrainOutput(global_step=12, training_loss=2.0658821066220603, metrics={'train_runtime': 65.5384, 'train_samples_per_second': 2.609, 'train_steps_per_second': 0.183, 'total_flos': 22230597703680.0, 'train_loss': 2.0658821066220603, 'epoch': 3.0})


## 7. Save the adapter

In [9]:
model.save_pretrained("finance_stage1_adapter")
tokenizer.save_pretrained("finance_stage1_adapter")
print("Stage 1 adapter saved to finance_stage1_adapter/")

# Optional: zip and download, or push to Hugging Face Hub for use in Stage 2
# from huggingface_hub import HfApi
# model.push_to_hub("your-username/finance-stage1-adapter", token="hf_...")

Unsloth: Restored added_tokens_decoder metadata in finance_stage1_adapter/tokenizer_config.json.


Stage 1 adapter saved to finance_stage1_adapter/


In [11]:
from google.colab import files
import shutil

shutil.make_archive("finance_stage1_adapter", "zip", "finance_stage1_adapter/")
files.download("finance_stage1_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Test the model after non-instruction fine-tuning
We give it a domain phrase prefix and let it freely continue — this checks whether it has absorbed finance vocabulary, not whether it follows instructions (that's Stage 2).

In [10]:
FastLanguageModel.for_inference(model)

test_prefixes = [
    "A fixed deposit is",
    "The interest-free period on a credit card",
    "To improve your credit score, you should",
    "A SIP allows an investor to",
]

for prefix in test_prefixes:
    inputs = tokenizer(prefix, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=60, do_sample=True, temperature=0.7, top_p=0.9)
    print("PROMPT:", prefix)
    print("CONTINUATION:", tokenizer.decode(out[0], skip_special_tokens=True))
    print("-" * 80)

Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=60

PROMPT: A fixed deposit is
CONTINUATION: A fixed deposit is a type of savings account that earns interest on a fixed percentage rate, typically stated as a percentage of the initial deposit amount, compounded monthly. Unlike other savings accounts, fixed deposits do not have a maturity date, meaning they are withdrawn early without penalty.
--------------------------------------------------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: The interest-free period on a credit card
CONTINUATION: The interest-free period on a credit card typically expires after a certain number of days, after which the full balance is typically due.
--------------------------------------------------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: To improve your credit score, you should
CONTINUATION: To improve your credit score, you should regularly review your credit report and make payments on time.
--------------------------------------------------------------------------------
PROMPT: A SIP allows an investor to
CONTINUATION: A SIP allows an investor to purchase shares of a company at a predetermined price, and receive dividends on those shares when the company earns profit.
--------------------------------------------------------------------------------


### Observation
After Stage 1, the model should generate continuations that *sound* financial (correct terminology, plausible domain phrasing) even though it has not yet learned to answer questions directly — that ability comes from Stage 2 instruction fine-tuning, which continues from `finance_stage1_adapter/`.